In [120]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import scipy as sp
%matplotlib qt
from continuum_solver import ContinuumSolver, TimeDependentSolver, clean_input
%load_ext autoreload
%autoreload 2

torch.backends.cuda.preferred_linalg_library("magma")
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
# set default datatype of tensors
DTYPE = torch.complex128

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [121]:
def harmonic_oscillator(x, x_max=5):
    return (1/2)*(x - x_max/2)**2

def mirror_cavity(x, g_0, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    a_mirror = 2 * np.cos(dx) + g_0 * np.sin(dx)
    r = (a_mirror - np.sqrt(a_mirror**2 -  4)) / 2
    g_defect = (2 * r - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def cavity(x, x_min, x_max, x_steps):
    dx = (x_max + x_min) / x_steps
    g_0 = 2 * (2 - np.cos(dx)) / np.sin(dx)
    g_defect = (2 * (2 - np.sqrt(3)) - 2 * np.cos(dx)) / np.sin(dx)

    output = torch.tensor([g_0], device=DEVICE, dtype=DTYPE).repeat(x.shape)
    output[x_steps // 2] = g_defect

    return output

def potential_fourier(x, x_min, x_max, x_steps, f_steps_min):
    V = torch.zeros((x_steps, 2 * f_steps_min + 1), device=DEVICE, dtype=DTYPE)
    V[:, f_steps_min] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))
    V[:, f_steps_min + 1] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))
    V[:, f_steps_min - 1] = 0.8 * torch.cos(2 * torch.pi * x / (x_max - x_min))

    return V


In [122]:
x_max = 2
x_steps = 128
f_steps_min = 1
epsilon = 2e-10
k_vals = torch.linspace(0, 2*torch.pi, 3, device=DEVICE)
energies = torch.linspace(0, 2, 200, device=DEVICE, dtype=DTYPE)

solver_ho = ContinuumSolver(
    lambda x: harmonic_oscillator(x, x_max=x_max),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_mirror = ContinuumSolver(
    lambda x: cavity(x, 0, x_max, x_steps),
    x_min=0,
    x_max=x_max,
    x_steps=x_steps
)
solver_time_dep = TimeDependentSolver(
    lambda x, f: potential_fourier(
        x, x_min=0, x_max=x_max, x_steps=x_steps, f_steps_min=f
    ),
    x_min=0.,
    x_max=x_max,
    x_steps=x_steps,
    f_steps_min=f_steps_min,
    omega=1
)

In [123]:
# output_lower_tri = solver_time_dep.collapse_a_exp(energies, "lower_tri")

In [124]:
# loss_lower_tri = solver_time_dep.loss(energies, clean_input(0), "lower_tri")
# loss_matrix = solver_time_dep.loss(energies, clean_input(0), "matrix_exp")
solver_time_dep.plot_loss(
    energies,
    torch.linspace(-torch.pi, torch.pi, 3, device=DEVICE),
    "matrix_exp"
)

C:\Users\surya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\cbook.py:1719: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
C:\Users\surya\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\cbook.py:1355: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


In [126]:
# fig, ax = plt.subplots(figsize=(10, 8))

# E_cpu = energies.cpu()

# ax.plot(E_cpu, loss_lower_tri.cpu(), label="Lower_tri Loss")
# ax.plot(E_cpu, loss_matrix.cpu(), label="Matrix_exp Loss")

# ax.legend()
# ax.grid()
# ax.set_title("Losses vs. Energy Plot")

# ax.set_xlabel("Energy")
# ax.set_ylabel("Loss")

# plt.show()

In [127]:
# eye = torch.eye(solver_time_dep.f_steps, device=DEVICE, dtype=DTYPE)
# fourier_coeffs = solver_time_dep.V(solver_time_dep.x_vals, solver_time_dep.f_steps_min)[:, solver_time_dep.f_steps_min]
# V_shape = solver_time_dep.x_vals.shape + solver_time_dep.f_vals.shape